## Notebook 5: AgentCore Identity and Gateway for the Sub-Agent Tools

This notebook puts every sub-agent tool behind a managed OAuth endpoint: one **AgentCore Gateway** fronted by **Cognito** Identity, with four Lambda-backed targets that package the existing `tools/*.py` modules. Gateway federates sub-agent tools while Code Interpreter stays a direct client call, so four tool modules move behind Gateway targets while `tools/code_interpreter_tool.py` stays on the agent side.

The notebook is split in two halves. The first half stands up the Lambda, the Cognito authorizer, and the Gateway. The second half registers the four targets and rewires the customer/order, diagnosis, and cost & recommendation agents to consume them via `McpToolset`.

### Architecture

![Architecture with Identity and Gateway](./images/architecture_with_gateway_v2.png)

Three AgentCore primitives now wire into the system, each attaching in a different place:

| Primitive | Attaches to | Transport | Why |
|---|---|---|---|
| **Memory** | `Runner` and `orchestrator_agent` | ADK `BaseMemoryService` over `MemoryClient` | Continuity is orchestrator-scoped, so memory attaches once at the top. |
| **Code Interpreter** | `cost_recommendation_agent` only | Direct `code_session()` client | Sandbox execution is its own AgentCore primitive, not an MCP tool. |
| **Identity + Gateway** | All three specialists | Cognito JWT, MCP JSON-RPC over HTTPS, `McpToolset` | Tool serving is a separate concern from agent authoring; one Cognito-protected endpoint federates every sub-agent's tools and gives a stable surface for Runtime-deployed specialists later on. |

The orchestrator's tool surface (3 sibling `AgentTool`s plus `preload_memory`) is unchanged from N4 v2. Only the underlying tools moved: each specialist's `tools=[func1, func2, ...]` list is replaced by `tools=[<agent>_toolset()]`, and the `McpToolset` opens a streaming MCP JSON-RPC session against the Gateway URL with a Cognito Bearer token. Same callable interface, different transport.

When a specialist calls a tool, the Gateway authenticates the token, looks up the requested target, and invokes the backing Lambda. The Lambda reads `context.client_context.custom["bedrockAgentCoreToolName"]`, splits on `___` to recover `{target_name, tool_name}`, resolves `target_name` to a tool module, and calls the matching function. `cost_recommendation_agent` still calls `execute_python` directly, since only MCP-served tool modules sit behind Gateway.

All 11 tools ship in a single `CustomerSupportTools` Lambda, since every tool module already imports the same shared JSON data layer and going wider with one Lambda per target would duplicate IAM and deployment four times for no benefit. Each Gateway target declares a subset of the tool schemas and points at the same Lambda ARN:

| Gateway target | Backing module | Tools exposed |
|---|---|---|
| `customer-order-tools` | `tools/customer_order_tools.py` | `get_customer_profile`, `get_order_details`, `verify_purchase` |
| `diagnosis-tools` | `tools/diagnosis_tools.py` | `search_knowledge_base`, `get_known_defects`, `web_search` |
| `eligibility-tools` | `tools/eligibility_tools.py` | `check_warranty_coverage`, `get_repair_options`, `check_replacement_inventory`, `get_return_eligibility` |
| `recommendation-tools` | `tools/recommendation_tools.py` | `get_repair_quote` |

The eligibility target is registered now even though no agent consumes it yet, so when the next notebook's attendees write `agents/eligibility_agent.py` the Gateway endpoint already exists. Gateway requires an OAuth-issued JWT on every call. The starter toolkit's `GatewayClient.create_oauth_authorizer_with_cognito` provisions a Cognito user pool with a `client_credentials` app client, a hosted domain, and an `{GatewayName}/invoke` resource-server scope in one call.

### Per-request auth flow

![Gateway auth flow](images/gateway_auth_flow.png)

`shared/auth.py` (set up later in the notebook) holds a cached Cognito `client_credentials` JWT with a 55-minute TTL, and the sub-agent's `McpToolset` presents that JWT as a `Bearer` token on every MCP JSON-RPC request. On `initialize` the Gateway issues an `Mcp-Session-Id` header that must be echoed on subsequent `tools/call` requests within the same session. The Gateway validates the JWT against the Cognito discovery URL, then invokes the shared `CustomerSupportTools` Lambda with `client_context.custom["bedrockAgentCoreToolName"]` set to `{target}___{tool}`. The handler splits on `___`, dispatches to the matching `tools/*.py` function, and returns the result back up the chain. The notebook exercises every one of these boundaries through the steps that follow.

### Prerequisites

- The previous notebook must have been run in this workspace, since modules under `shared/`, `tools/`, `agents/`, `scenarios/` are imported directly and `memory_id` is read from SSM.
- AWS credentials for `us-west-2` with permissions for IAM (`CreateRole`, `GetRole`, `AttachRolePolicy`), Lambda (`CreateFunction`, `UpdateFunctionCode`, `GetFunction`, `GetFunctionConfiguration`), Cognito (`cognito-idp:*` on the pool this notebook creates), AgentCore Gateway control-plane (`CreateGateway`, `CreateGatewayTarget`, `GetGateway`, `ListGateways`), and SSM read/write on `/agentcore-workshop/*`.
- Bedrock model access for Claude Haiku 4.5 and Claude Sonnet 4.6 in `us-west-2`.

This notebook provisions four kinds of AWS resources, all fixed-name and idempotent:

| Resource | Name | SSM key |
|---|---|---|
| IAM role for Lambda | `CustomerSupportToolsLambdaRole` | `iam/lambda_role_arn` |
| Lambda function | `CustomerSupportTools` | `lambda/customer_support_tools_arn` |
| Cognito user pool and app client | auto-named by `GatewayClient` | `cognito/token_endpoint`, `cognito/client_id`, `cognito/client_secret`, `cognito/scope` |
| Gateway and four targets | `CustomerSupportGateway` | `gateway/gateway_url`, `gateway/gateway_id`, `gateway/targets/{target_name}_id` |

Each setup cell checks for the resource first and reuses it when present, so re-running never creates duplicates.

### Step 0: Environment

The imports below bring in the building blocks this notebook uses: `boto3` for IAM, Lambda, and Cognito control-plane calls; `zipfile` and `io` for packaging the Lambda code; the starter-toolkit's `GatewayClient` for Cognito and Gateway provisioning; and the SSM helpers from `shared/config.py` so every resource ID lands under `/agentcore-workshop/`.

In [64]:
%pip install "mcp >= 1.15" --upgrade

Note: you may need to restart the kernel to use updated packages.


In [1]:
import boto3
import io
import json
import time
import zipfile
from pathlib import Path

from bedrock_agentcore_starter_toolkit.operations.gateway.client import GatewayClient

import shared.config as _shared_config
from shared.config import AWS_REGION, SSM_PREFIX, get_ssm_parameter, put_ssm_parameter

# Anchor every path operation to the workshop root -- computed from the shared
# package location rather than cwd, so the notebook works regardless of where
# Jupyter is launched from.
WORKSHOP_ROOT = Path(_shared_config.__file__).resolve().parent.parent

# %%writefile does NOT auto-create parent directories; make sure the directory
# exists before Step 3 writes gateway/lambda_handler.py.
(WORKSHOP_ROOT / "gateway").mkdir(exist_ok=True)

print("AWS_REGION:", AWS_REGION)
print("SSM_PREFIX:", SSM_PREFIX)
print("WORKSHOP_ROOT:", WORKSHOP_ROOT)
print("GatewayClient:", GatewayClient)


AWS_REGION: us-west-2
SSM_PREFIX: /agentcore-workshop
WORKSHOP_ROOT: /home/sagemaker-user/bedrock-agentcore-workshop
GatewayClient: <class 'bedrock_agentcore_starter_toolkit.operations.gateway.client.GatewayClient'>


### Step 1: Name the Resources and Declare the Tool Schemas

Each attendee runs in their own account, so there are no cross-tenant naming collisions and pinning the names as constants up front makes the idempotency checks a simple "try get-by-name, on NotFound create" pattern. The `TARGETS` dictionary below is the authoritative source for what each Gateway target exposes (four targets, 11 tools in total). Each tool entry is a `{name, description, inputSchema}` dict in the shape MCP expects, which is the same shape the starter-toolkit's `create_mcp_gateway_target` accepts under `target_payload.toolSchema.inlinePayload`.

In [2]:
LAMBDA_NAME = "CustomerSupportTools"
LAMBDA_ROLE_NAME = "CustomerSupportToolsLambdaRole"
GATEWAY_NAME = "CustomerSupportGateway"

TARGETS = {
    "customer-order-tools": [
        {
            "name": "get_customer_profile",
            "description": (
                "Retrieve customer profile by customer ID. Returns name, account tier, "
                "and contact info (no raw financial data)."
            ),
            "inputSchema": {
                "type": "object",
                "properties": {
                    "customer_id": {
                        "type": "string",
                        "description": "Customer identifier, e.g. C001.",
                    },
                },
                "required": ["customer_id"],
            },
        },
        {
            "name": "get_order_details",
            "description": (
                "Retrieve full order details by order ID. Returns product info, "
                "purchase date, and delivery status."
            ),
            "inputSchema": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "Order identifier, e.g. ORD-2026-0342.",
                    },
                },
                "required": ["order_id"],
            },
        },
        {
            "name": "verify_purchase",
            "description": (
                "Confirm that an order belongs to a specific customer. Returns "
                "verified status, days since purchase, and days since delivery."
            ),
            "inputSchema": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string"},
                    "customer_id": {"type": "string"},
                },
                "required": ["order_id", "customer_id"],
            },
        },
    ],
    "diagnosis-tools": [
        {
            "name": "search_knowledge_base",
            "description": (
                "Semantic search over the Bedrock Knowledge Base of product manuals, "
                "setup guides, firmware changelogs, maintenance instructions, and "
                "known issue advisories. Optionally filter by product SKU for precision."
            ),
            "inputSchema": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Symptom description in natural language.",
                    },
                    "product_sku": {
                        "type": "string",
                        "description": "Optional SKU filter.",
                    },
                },
                "required": ["query"],
            },
        },
        {
            "name": "get_known_defects",
            "description": (
                "Look up known defects and service advisories for a specific product "
                "SKU. Returns all active defect records including fix availability."
            ),
            "inputSchema": {
                "type": "object",
                "properties": {
                    "product_sku": {
                        "type": "string",
                        "description": "Exact SKU from the order record.",
                    },
                },
                "required": ["product_sku"],
            },
        },
        {
            "name": "web_search",
            "description": (
                "Search the web for technical information not found in the knowledge "
                "base. Used as a fallback when KB search returns no relevant results."
            ),
            "inputSchema": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "max_results": {
                        "type": "integer",
                        "description": "Maximum number of articles to return. Defaults to 3.",
                    },
                },
                "required": ["query"],
            },
        },
    ],
    "eligibility-tools": [
        {
            "name": "check_warranty_coverage",
            "description": (
                "Check whether the diagnosed defect is covered under manufacturer "
                "warranty. Returns coverage status, reason, and remaining warranty days."
            ),
            "inputSchema": {
                "type": "object",
                "properties": {
                    "product_sku": {"type": "string"},
                    "days_since_purchase": {"type": "integer"},
                    "defect_type": {
                        "type": "string",
                        "description": "e.g. firmware_bug, hardware_failure, physical_damage.",
                    },
                    "account_tier": {
                        "type": "string",
                        "description": "Customer account tier, e.g. premium or standard.",
                    },
                },
                "required": [
                    "product_sku",
                    "days_since_purchase",
                    "defect_type",
                    "account_tier",
                ],
            },
        },
        {
            "name": "get_repair_options",
            "description": (
                "Get available repair options for a product and defect type. If "
                "warranty_covered is False, only returns paid repair options."
            ),
            "inputSchema": {
                "type": "object",
                "properties": {
                    "product_sku": {"type": "string"},
                    "defect_type": {"type": "string"},
                    "warranty_covered": {"type": "boolean"},
                },
                "required": ["product_sku", "defect_type", "warranty_covered"],
            },
        },
        {
            "name": "check_replacement_inventory",
            "description": (
                "Check if a replacement unit of the same product is available in stock. "
                "Applies tier-based shipping uplifts when present."
            ),
            "inputSchema": {
                "type": "object",
                "properties": {
                    "product_sku": {"type": "string"},
                    "account_tier": {
                        "type": "string",
                        "description": "Customer account tier, e.g. premium or standard.",
                    },
                },
                "required": ["product_sku", "account_tier"],
            },
        },
        {
            "name": "get_return_eligibility",
            "description": (
                "Check whether the order is eligible for a return and refund. Returns "
                "eligibility, refund amount, and any conditions."
            ),
            "inputSchema": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string"},
                    "category": {
                        "type": "string",
                        "description": "Product category, e.g. laptops.",
                    },
                    "days_since_delivery": {"type": "integer"},
                },
                "required": ["order_id", "category", "days_since_delivery"],
            },
        },
    ],
    "recommendation-tools": [
        {
            "name": "get_repair_quote",
            "description": (
                "Get current repair quote from the authorized service network. Returns "
                "labor and parts costs for the specific defect type."
            ),
            "inputSchema": {
                "type": "object",
                "properties": {
                    "product_sku": {"type": "string"},
                    "defect_type": {"type": "string"},
                },
                "required": ["product_sku", "defect_type"],
            },
        },
    ],
}

print("Gateway:", GATEWAY_NAME)
print("Lambda:", LAMBDA_NAME)
print("Lambda role:", LAMBDA_ROLE_NAME)
print("Targets:", list(TARGETS.keys()))
print("Total tools to register:", sum(len(tools) for tools in TARGETS.values()))

Gateway: CustomerSupportGateway
Lambda: CustomerSupportTools
Lambda role: CustomerSupportToolsLambdaRole
Targets: ['customer-order-tools', 'diagnosis-tools', 'eligibility-tools', 'recommendation-tools']
Total tools to register: 11


### Step 2: Create the Lambda Execution Role

The Lambda needs an IAM role with two ingredients: a trust policy that lets the Lambda service assume it (the standard `lambda.amazonaws.com` principal), and the managed policy `AWSLambdaBasicExecutionRole` so the function can write to CloudWatch Logs. The tool code reads only in-memory JSON bundled into the deployment package, so no other AWS permissions are needed.

The cell is idempotent: a `get_role` call either succeeds (the role already exists) or raises `NoSuchEntity`. On creation we sleep for half a minute to let the role propagate, since `create_function` sometimes races ahead of IAM's eventual consistency.

In [3]:
iam_client = boto3.client("iam", region_name=AWS_REGION)


def ensure_lambda_role() -> str:
    """Return the ARN of the Lambda execution role, creating it on first run."""
    try:
        existing = iam_client.get_role(RoleName=LAMBDA_ROLE_NAME)
        role_arn = existing["Role"]["Arn"]
        print(f"IAM role {LAMBDA_ROLE_NAME} already exists: {role_arn}")
        return role_arn
    except iam_client.exceptions.NoSuchEntityException:
        pass

    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {"Service": "lambda.amazonaws.com"},
                "Action": "sts:AssumeRole",
            }
        ],
    }
    created = iam_client.create_role(
        RoleName=LAMBDA_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Execution role for the CustomerSupportTools Lambda (workshop).",
    )
    role_arn = created["Role"]["Arn"]
    iam_client.attach_role_policy(
        RoleName=LAMBDA_ROLE_NAME,
        PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
    )
    print(f"Created IAM role {LAMBDA_ROLE_NAME}: {role_arn}")
    print("Waiting 30s for IAM role propagation before Lambda creation...")
    time.sleep(30)
    return role_arn


lambda_role_arn = ensure_lambda_role()
put_ssm_parameter("iam/lambda_role_arn", lambda_role_arn)
print(f"Persisted to SSM: {SSM_PREFIX}/iam/lambda_role_arn")

IAM role CustomerSupportToolsLambdaRole already exists: arn:aws:iam::497029937664:role/CustomerSupportToolsLambdaRole
Persisted to SSM: /agentcore-workshop/iam/lambda_role_arn


### Step 3: Write the Lambda Handler

The Gateway invokes one Lambda for every tool call and puts the full tool name in the form `{target_name}___{tool_name}` into `context.client_context.custom["bedrockAgentCoreToolName"]` (the separator is a literal triple underscore). That single field is all the routing signal the handler needs: split on `___`, use the left half to pick a tool module, and the right half to pick the function inside it.

| `target_name` | Module imported |
|---|---|
| `customer-order-tools` | `tools.customer_order_tools` |
| `diagnosis-tools` | `tools.diagnosis_tools` |
| `eligibility-tools` | `tools.eligibility_tools` |
| `recommendation-tools` | `tools.recommendation_tools` |

The handler is stdlib-only plus the local `tools.*` imports. Everything the tools touch (`shared.config`, `shared.data`, the `data/*.json` files) is bundled into the deployment package, so the Lambda never hits the network except through AWS-internal APIs. Gateway can pass tool arguments either as a top-level dict in `event` or inside a JSON-encoded `body` field, so the handler normalizes both shapes into a plain `kwargs` dict before dispatching.

In [4]:
%%writefile gateway/lambda_handler.py
"""Lambda handler for the CustomerSupportTools Gateway target.

One Lambda backs all four Gateway targets. The Gateway embeds the fully
qualified tool name -- ``{target_name}___{tool_name}`` -- in
``context.client_context.custom['bedrockAgentCoreToolName']``. The handler
splits that on ``___``, resolves the left half to a tool module, looks up
the right half as a function attribute on that module, and calls it with the
event arguments.

Stdlib-only plus local ``tools.*`` / ``shared.*`` imports; no agent framework
dependencies ship inside this package.
"""
import json
import importlib

TARGET_TO_MODULE = {
    "customer-order-tools": "tools.customer_order_tools",
    "diagnosis-tools": "tools.diagnosis_tools",
    "eligibility-tools": "tools.eligibility_tools",
    "recommendation-tools": "tools.recommendation_tools",
}


def _extract_kwargs(event):
    """Return a plain dict of tool arguments from either event shape.

    Gateway may pass arguments as the top-level event or as a JSON-encoded
    ``body`` field. Handle both so the tool functions themselves stay unchanged.
    """
    if not isinstance(event, dict):
        return {}
    body = event.get("body")
    if isinstance(body, str):
        try:
            parsed = json.loads(body)
            return parsed if isinstance(parsed, dict) else {}
        except json.JSONDecodeError:
            return {}
    if isinstance(body, dict):
        return body
    # No ``body`` field: treat the event itself as the argument dict, minus
    # any Lambda / API Gateway metadata fields a caller might have slipped in.
    return {k: v for k, v in event.items() if k != "body"}


def lambda_handler(event, context):
    try:
        extended_tool_name = context.client_context.custom["bedrockAgentCoreToolName"]
    except (AttributeError, KeyError, TypeError) as exc:
        return {
            "statusCode": 400,
            "body": f"Missing bedrockAgentCoreToolName in client context: {exc}",
        }

    if "___" not in extended_tool_name:
        return {
            "statusCode": 400,
            "body": f"Malformed tool name (expected target___tool): {extended_tool_name}",
        }

    target_name, tool_name = extended_tool_name.split("___", 1)
    module_path = TARGET_TO_MODULE.get(target_name)
    if module_path is None:
        return {
            "statusCode": 400,
            "body": f"Unknown target: {target_name}",
        }

    try:
        module = importlib.import_module(module_path)
    except ImportError as exc:
        return {
            "statusCode": 500,
            "body": f"Failed to import {module_path}: {exc}",
        }

    tool_fn = getattr(module, tool_name, None)
    if tool_fn is None or not callable(tool_fn):
        return {
            "statusCode": 400,
            "body": f"Unknown tool {tool_name} in {module_path}",
        }

    kwargs = _extract_kwargs(event)
    try:
        result = tool_fn(**kwargs)
    except TypeError as exc:
        return {
            "statusCode": 400,
            "body": f"Invalid arguments for {tool_name}: {exc}",
        }
    except Exception as exc:  # noqa: BLE001 -- Gateway expects a body, never a raise
        return {
            "statusCode": 500,
            "body": f"Tool {tool_name} raised: {exc}",
        }

    return {
        "statusCode": 200,
        "body": json.dumps(result),
    }


Overwriting gateway/lambda_handler.py


### Step 4: Package and Deploy the Lambda

The deployment package is built in memory and contains exactly what the handler needs at runtime:

- `lambda_function.py` at the archive root (the handler from Step 3, renamed so the Lambda runtime's default `lambda_function.lambda_handler` target resolves correctly).
- The four Gateway-bound tool modules under `tools/`, plus a `tools/__init__.py`. `tools/code_interpreter_tool.py` is excluded because Code Interpreter is a direct client call.
- The data-loader layer the tools import: `shared/__init__.py`, `shared/config.py`, `shared/data.py`.
- The seven JSON files under `data/` that the tools read.

Nothing else ships, since `bedrock_agentcore`, `google-adk`, and `litellm` are agent-side dependencies and keeping them out keeps the deployment zip in the single-digit-KB range. Idempotency is handled by the Lambda API itself: on first run `create_function` succeeds, and on subsequent runs it raises `ResourceConflictException` which the cell catches and converts into `update_function_code`. After every deploy the cell waits for the function to reach `Active` before returning, which avoids a race where target registration tries to resolve a Lambda ARN that is still `Pending`.

In [5]:
# Files that go into the deployment package. Paths are relative to WORKSHOP_ROOT.
SHARED_FILES = [
    Path("shared") / "__init__.py",
    Path("shared") / "config.py",
    Path("shared") / "data.py",
]
TOOL_FILES = [
    Path("tools") / "customer_order_tools.py",
    Path("tools") / "diagnosis_tools.py",
    Path("tools") / "eligibility_tools.py",
    Path("tools") / "recommendation_tools.py",
]
HANDLER_FILE = Path("gateway") / "lambda_handler.py"
DATA_DIR = Path("data")


def _build_deployment_zip() -> bytes:
    """Assemble the Lambda deployment package in memory and return its bytes."""
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as archive:
        # Handler -> lambda_function.py at archive root
        archive.write(
            WORKSHOP_ROOT / HANDLER_FILE, arcname="lambda_function.py"
        )
        # tools/ package (explicit __init__.py so the import path is a package)
        archive.writestr("tools/__init__.py", "")
        for tool_path in TOOL_FILES:
            archive.write(WORKSHOP_ROOT / tool_path, arcname=str(tool_path))
        # shared/ package
        for shared_path in SHARED_FILES:
            archive.write(
                WORKSHOP_ROOT / shared_path, arcname=str(shared_path)
            )
        # data/ JSON files
        for json_file in sorted((WORKSHOP_ROOT / DATA_DIR).glob("*.json")):
            archive.write(json_file, arcname=str(DATA_DIR / json_file.name))
    buffer.seek(0)
    return buffer.read()


def _wait_for_active(client, function_name: str) -> None:
    """Block until Lambda reports ``State == Active`` (or raise after a minute)."""
    for _ in range(30):
        cfg = client.get_function_configuration(FunctionName=function_name)
        state = cfg.get("State")
        last_status = cfg.get("LastUpdateStatus", "Successful")
        if state == "Active" and last_status == "Successful":
            return
        if state == "Failed" or last_status == "Failed":
            raise RuntimeError(
                f"Lambda {function_name} entered failed state: {cfg}"
            )
        time.sleep(2)
    raise TimeoutError(f"Lambda {function_name} did not become Active within 60s")


lambda_client = boto3.client("lambda", region_name=AWS_REGION)
zip_bytes = _build_deployment_zip()
print(f"Built deployment package: {len(zip_bytes)} bytes")

try:
    created = lambda_client.create_function(
        FunctionName=LAMBDA_NAME,
        Runtime="python3.12",
        Role=lambda_role_arn,
        Handler="lambda_function.lambda_handler",
        Code={"ZipFile": zip_bytes},
        Timeout=30,
        MemorySize=256,
        Description="Customer support tools backing the Gateway targets (workshop).",
    )
    lambda_arn = created["FunctionArn"]
    print(f"Created Lambda {LAMBDA_NAME}: {lambda_arn}")
except lambda_client.exceptions.ResourceConflictException:
    updated = lambda_client.update_function_code(
        FunctionName=LAMBDA_NAME,
        ZipFile=zip_bytes,
    )
    lambda_arn = updated["FunctionArn"]
    print(f"Lambda {LAMBDA_NAME} already existed; pushed updated code.")

_wait_for_active(lambda_client, LAMBDA_NAME)
put_ssm_parameter("lambda/customer_support_tools_arn", lambda_arn)
print(f"Lambda is Active. Persisted ARN to {SSM_PREFIX}/lambda/customer_support_tools_arn")

Built deployment package: 11265 bytes
Lambda CustomerSupportTools already existed; pushed updated code.
Lambda is Active. Persisted ARN to /agentcore-workshop/lambda/customer_support_tools_arn


### Step 5: Create the Cognito OAuth Authorizer

Gateway expects a JWT from an OAuth authorizer, by convention a Cognito user pool with a machine-to-machine app client configured for the `client_credentials` grant. `GatewayClient.create_oauth_authorizer_with_cognito(gateway_name)` provisions the full stack in one call (user pool, user-pool domain, resource server with an `invoke` scope, and an app client with secret), waiting the roughly sixty seconds Cognito needs for DNS propagation before returning. The return value splits into:

- `authorizer_config`, the structured blob `create_mcp_gateway` consumes (Cognito issuer, allowed client IDs, scope).
- `client_info`, the credentials the agent side needs to fetch access tokens (`token_endpoint`, `client_id`, `client_secret`, `scope`, `user_pool_id`, `domain_prefix`).

The cell is idempotent: it first tries to read `cognito/client_id` from SSM, and a hit means the pool already exists in this account and is reconstituted from the persisted values. Otherwise it goes through the full create path and persists every field under `/agentcore-workshop/cognito/`, with `client_secret` encrypted as a `SecureString`.

In [6]:
gateway_client = GatewayClient(region_name=AWS_REGION)
ssm_client = boto3.client("ssm", region_name=AWS_REGION)

_COGNITO_KEYS = [
    "cognito/client_id",
    "cognito/client_secret",
    "cognito/token_endpoint",
    "cognito/scope",
    "cognito/user_pool_id",
]


def _discovery_url(user_pool_id: str) -> str:
    """Return the OpenID Connect discovery URL Gateway uses to validate JWTs.

    This is the Cognito user-pool URL (NOT the hosted-UI domain) -- matches
    what GatewayClient.create_oauth_authorizer_with_cognito returns.
    """
    return (
        f"https://cognito-idp.{AWS_REGION}.amazonaws.com/"
        f"{user_pool_id}/.well-known/openid-configuration"
    )


def ensure_cognito_authorizer():
    """Return ``(authorizer_config, client_info)`` for the Gateway.

    Reuses an existing pool whose details are persisted under
    ``/agentcore-workshop/cognito/`` when present; otherwise creates a fresh
    Cognito stack via the starter toolkit and persists every field.
    """
    # Check all required keys up front. Partial SSM state is an error -- silently
    # creating a duplicate Cognito pool would orphan the old one.
    present = {}
    missing = []
    for key in _COGNITO_KEYS:
        try:
            present[key] = get_ssm_parameter(key)
        except ssm_client.exceptions.ParameterNotFound:
            missing.append(key)

    if missing and present:
        raise RuntimeError(
            f"Partial Cognito SSM state detected. Missing: {missing}. "
            f"Present: {sorted(present)}. Either restore the missing keys or "
            f"delete the present ones (aws ssm delete-parameters) so this "
            f"cell can recreate the Cognito pool cleanly."
        )

    if not missing:
        existing_info = {
            "client_id": present["cognito/client_id"],
            "client_secret": present["cognito/client_secret"],
            "token_endpoint": present["cognito/token_endpoint"],
            "scope": present["cognito/scope"],
            "user_pool_id": present["cognito/user_pool_id"],
        }
        existing_config = {
            "customJWTAuthorizer": {
                "allowedClients": [existing_info["client_id"]],
                "discoveryUrl": _discovery_url(existing_info["user_pool_id"]),
            }
        }
        print(f"Cognito authorizer already configured (client_id={existing_info['client_id']}).")
        return existing_config, existing_info

    print("Creating Cognito OAuth authorizer (waits ~60s for DNS propagation)...")
    result = gateway_client.create_oauth_authorizer_with_cognito(GATEWAY_NAME)
    authorizer_config = result["authorizer_config"]
    client_info = result["client_info"]

    put_ssm_parameter("cognito/token_endpoint", client_info["token_endpoint"])
    put_ssm_parameter("cognito/client_id", client_info["client_id"])
    put_ssm_parameter(
        "cognito/client_secret", client_info["client_secret"], encrypted=True
    )
    put_ssm_parameter("cognito/scope", client_info.get("scope", ""))
    put_ssm_parameter("cognito/user_pool_id", client_info["user_pool_id"])
    print(
        f"Cognito authorizer created; client_id={client_info['client_id']}, "
        f"user_pool_id={client_info['user_pool_id']}. Persisted to {SSM_PREFIX}/cognito/*."
    )
    return authorizer_config, client_info


authorizer_config, cognito_client_info = ensure_cognito_authorizer()
print("token_endpoint:", cognito_client_info["token_endpoint"])
print("scope:", cognito_client_info.get("scope", ""))


Cognito authorizer already configured (client_id=6mb091k03ofogd3gdgabvhb3q4).
token_endpoint: https://agentcore-ca75e7cd.auth.us-west-2.amazoncognito.com/oauth2/token
scope: CustomerSupportGateway/invoke


### Step 6: Create the MCP Gateway

With the Cognito authorizer in hand, the Gateway is a single `GatewayClient.create_mcp_gateway` call that returns a `gatewayId` and a `gatewayUrl`, the MCP endpoint that `McpToolset` will point at later. The starter toolkit auto-creates the Gateway execution role (`AgentCoreGatewayExecutionRole`) with Lambda-invoke permissions on the account, so any Lambda ARN registered as a target becomes invokable without per-Lambda permission grants.

The cell is idempotent on SSM: a `gateway_url` already at `/agentcore-workshop/gateway/gateway_url` means the Gateway exists from a prior run and is reused verbatim. Otherwise we call `create_mcp_gateway` with the authorizer config from Step 5 and persist the returned ID and URL.

In [7]:
def ensure_gateway():
    """Return a ``{"gatewayId": ..., "gatewayUrl": ...}`` dict for the Gateway.

    Reuses an existing Gateway whose id + url are in SSM; otherwise creates it
    via the starter toolkit using the Cognito authorizer from Step 5.
    """
    try:
        gateway_url = get_ssm_parameter("gateway/gateway_url")
        gateway_id = get_ssm_parameter("gateway/gateway_id")
        print(f"Gateway {GATEWAY_NAME} already exists (id={gateway_id}).")
        return {"gatewayId": gateway_id, "gatewayUrl": gateway_url}
    except ssm_client.exceptions.ParameterNotFound:
        pass

    print(f"Creating MCP Gateway: {GATEWAY_NAME}")
    gateway = gateway_client.create_mcp_gateway(
        name=GATEWAY_NAME,
        authorizer_config=authorizer_config,
    )
    gateway_id = gateway["gatewayId"]
    gateway_url = gateway["gatewayUrl"]
    put_ssm_parameter("gateway/gateway_url", gateway_url)
    put_ssm_parameter("gateway/gateway_id", gateway_id)
    print(f"Gateway created: id={gateway_id}")
    print(f"Gateway URL: {gateway_url}")
    return {"gatewayId": gateway_id, "gatewayUrl": gateway_url}


gateway = ensure_gateway()
print(f"Persisted Gateway id + url under {SSM_PREFIX}/gateway/")

Gateway CustomerSupportGateway already exists (id=customersupportgateway-rbnx6ivi4e).
Persisted Gateway id + url under /agentcore-workshop/gateway/


### Step 7: Register the Four Gateway Targets

The Gateway is live but serves zero tools until each `TARGETS` entry is registered as a target bound to the `CustomerSupportTools` Lambda ARN. Each `create_mcp_gateway_target` call takes the Lambda ARN plus the target's `toolSchema` (the `inlinePayload` list of `{name, description, inputSchema}` dicts assembled in Step 1) and returns a `targetId` that we persist under `/agentcore-workshop/gateway/targets/{target_name}_id`, so re-runs reuse the registration. Code Interpreter remains a direct client call and is deliberately not exposed as a Gateway target.

In [8]:
# Register each target, skipping any that are already recorded in SSM.
from botocore.exceptions import ClientError

gateway_id = gateway["gatewayId"]

# Harden this cell against standalone re-runs: if the in-memory lambda_arn
# is missing (attendee skipped Step 4), fall back to the persisted ARN.
try:
    lambda_arn
except NameError:
    lambda_arn = get_ssm_parameter("lambda/customer_support_tools_arn")

for target_name, tool_schemas in TARGETS.items():
    ssm_key = f"gateway/targets/{target_name}_id"
    try:
        existing_id = get_ssm_parameter(ssm_key)
        print(f"{target_name}: already registered (target_id={existing_id}), skipping")
        continue
    except ClientError as exc:
        if exc.response["Error"]["Code"] != "ParameterNotFound":
            raise

    print(f"{target_name}: registering {len(tool_schemas)} tool(s)...")
    target = gateway_client.create_mcp_gateway_target(
        gateway={"gatewayId": gateway_id},
        name=target_name,
        target_type="lambda",
        target_payload={
            "lambdaArn": lambda_arn,
            "toolSchema": {"inlinePayload": tool_schemas},
        },
    )
    target_id = target["targetId"]
    put_ssm_parameter(ssm_key, target_id)
    print(f"{target_name}: registered (target_id={target_id})")

print("\nAll four targets are registered on the Gateway.")

# Gateway needs a few seconds to propagate new target metadata before
# `tools/list` sees the tools. If any target was created in this run (not all
# reused from SSM), sleep briefly so downstream cells and agent imports see
# the full catalog. This mirrors the 30s IAM propagation wait in Step 2.
import time
time.sleep(30)
print("Waited 30s for Gateway target propagation; downstream McpToolset calls will see all tools.")


customer-order-tools: already registered (target_id=CS0YIOMQ7W), skipping
diagnosis-tools: already registered (target_id=S4CNPNKZWM), skipping
eligibility-tools: already registered (target_id=LSBPTM2KPY), skipping
recommendation-tools: already registered (target_id=F7EBOQY57X), skipping

All four targets are registered on the Gateway.
Waited 30s for Gateway target propagation; downstream McpToolset calls will see all tools.


### Step 8: Cognito Token Helper (`shared/auth.py`)

Every call the Gateway accepts carries a Bearer token issued by the Cognito app client. The agents fetch one token per session and reuse it. At construction time each `McpToolset` is handed a fresh token via `shared.auth.get_cognito_token()`. Two decisions drive this module:

1. Stdlib only. `urllib.request` replaces a `requests` dependency, which matters in the Runtime notebook where the sub-agent containers ship a minimal `requirements.txt` and the AgentCore starter toolkit is deliberately absent. Using the same helper locally and in the container keeps the auth path identical across environments.
2. An in-process cache with a 55-minute TTL. Cognito `client_credentials` tokens live for 60 minutes, so refreshing five minutes early avoids edge-of-expiry failures without needing refresh coordination between agents.

In [9]:
%%writefile shared/auth.py
# workshop/shared/auth.py
"""Cognito client_credentials token fetch with a 55-minute in-process cache.

Uses only the Python standard library so the same code runs locally and inside
the AgentCore Runtime containers built in N6.
"""
from __future__ import annotations

import base64
import json
import time
import urllib.parse
import urllib.request

from shared.config import get_ssm_parameter

_TOKEN_TTL_SECONDS = 55 * 60
_cache: dict[str, tuple[str, float]] = {}


def get_cognito_token() -> str:
    """Return a valid Cognito access token, refreshing at most every 55 minutes."""
    token_endpoint = get_ssm_parameter("cognito/token_endpoint")
    client_id = get_ssm_parameter("cognito/client_id")
    client_secret = get_ssm_parameter("cognito/client_secret")
    scope = get_ssm_parameter("cognito/scope")

    cached = _cache.get(client_id)
    if cached is not None:
        token, expires_at = cached
        if time.time() < expires_at:
            return token

    basic = base64.b64encode(f"{client_id}:{client_secret}".encode()).decode()
    body = urllib.parse.urlencode(
        {
            "grant_type": "client_credentials",
            "client_id": client_id,
            "client_secret": client_secret,
            "scope": scope,
        }
    ).encode()

    request = urllib.request.Request(
        token_endpoint,
        data=body,
        headers={
            "Authorization": f"Basic {basic}",
            "Content-Type": "application/x-www-form-urlencoded",
        },
    )
    with urllib.request.urlopen(request, timeout=15) as response:
        payload = json.loads(response.read().decode())

    access_token = payload["access_token"]
    _cache[client_id] = (access_token, time.time() + _TOKEN_TTL_SECONDS)
    return access_token

Overwriting shared/auth.py


### Step 9: Per-Agent `McpToolset` Factories (`shared/gateway.py`)

The Gateway exposes every registered tool from a single MCP endpoint and each tool's name is prefixed with its target (for example `customer-order-tools___get_customer_profile`). Handing an agent the raw toolset would let it see every target's tools, which defeats the auth-isolation design.

The fix is one factory per target, each returning an `McpToolset` whose `tool_filter` keeps only the tools whose names begin with that target's prefix. All four factories are exported, including `eligibility_toolset()` for the agent introduced in the next notebook, so attendees there write only an ADK agent file.

In [10]:
%%writefile shared/gateway.py
# workshop/shared/gateway.py
"""Per-target McpToolset factories for the four sub-agent tool groups.

Each factory opens a streaming MCP session to the shared Gateway URL with a
fresh Cognito Bearer token and filters the exposed tools to a single target's
prefix. Agents import only the factory they need.
"""
from __future__ import annotations

from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset

from shared.auth import get_cognito_token
from shared.config import get_ssm_parameter

_CUSTOMER_ORDER_TARGET = "customer-order-tools"
_DIAGNOSIS_TARGET = "diagnosis-tools"
_ELIGIBILITY_TARGET = "eligibility-tools"
_RECOMMENDATION_TARGET = "recommendation-tools"


def _make_toolset(target_name: str) -> McpToolset:
    gateway_url = get_ssm_parameter("gateway/gateway_url")
    token = get_cognito_token()
    prefix = f"{target_name}___"
    return McpToolset(
        connection_params=StreamableHTTPConnectionParams(
            url=gateway_url,
            headers={"Authorization": f"Bearer {token}"},
            timeout=30.0,
            sse_read_timeout=60.0,
        ),
        tool_filter=lambda tool, _ctx, _prefix=prefix: tool.name.startswith(_prefix),
    )


def customer_order_toolset() -> McpToolset:
    """Gateway-served tools for Agent 1 (Customer & Order)."""
    return _make_toolset(_CUSTOMER_ORDER_TARGET)


def diagnosis_toolset() -> McpToolset:
    """Gateway-served tools for Agent 2 (Diagnosis)."""
    return _make_toolset(_DIAGNOSIS_TARGET)


def eligibility_toolset() -> McpToolset:
    """Gateway-served tools for Agent 3 (Resolution Eligibility), consumed starting in N5."""
    return _make_toolset(_ELIGIBILITY_TARGET)


def recommendation_toolset() -> McpToolset:
    """Gateway-served tools for Agent 4 (Cost & Recommendation)."""
    return _make_toolset(_RECOMMENDATION_TARGET)

Overwriting shared/gateway.py


### Step 10: Rewire the Three Specialists to Use the Gateway

The orchestrator's tool surface (three sibling `AgentTool`s plus `preload_memory`) is unchanged. Only the underlying tools moved. The three agent modules from Notebook 1 imported their tools as Python functions from `tools/*.py`; with the Gateway in place those imports become a single toolset factory call (`<agent>_toolset()`), while system prompts, model selections, agent names, and `output_key` wiring stay the same. The cost & recommendation agent retains `execute_python` alongside `recommendation_toolset()`, since Code Interpreter is never routed through Gateway, and as the final stage it deliberately has no `output_key`.

In [11]:
%%writefile agents/customer_order_agent.py
# workshop/agents/customer_order_agent.py
from google.adk.agents import Agent

from shared.gateway import customer_order_toolset
from shared.models import HAIKU

SYSTEM_PROMPT = """You are the Customer & Order Agent for an electronics e-commerce support system.

Grounding rules (apply to every reply):
1. State only facts returned by your tools in this turn. If a tool did not return a value
   for a field, write "not found" for that field. Do not infer from product name,
   customer name, prior turns, or general knowledge.
2. Never invent customer IDs, order IDs, SKUs, dates, prices, statuses, or tier values.
3. Do not share raw PII (email, phone, address). Return only what downstream agents need:
   verified status, product name, product SKU, purchase date, days since delivery,
   order status, and customer tier.

Procedure for verifying a customer and order:
1. Look up the customer record using the customer ID.
2. Look up the order record using the order ID.
3. Confirm the order belongs to this customer. If it does not, return verified=false
   and stop -- do not return order details.
4. Return a single structured summary with the fields above. Be terse: one short
   labeled line per field is enough."""

customer_order_agent = Agent(
    name="customer_order_agent",
    model=HAIKU,
    description="Verifies customer identity and retrieves order details.",
    instruction=SYSTEM_PROMPT,
    tools=[customer_order_toolset()],
    output_key="customer_order",
)


Overwriting agents/customer_order_agent.py


In [12]:
%%writefile agents/diagnosis_agent.py
# workshop/agents/diagnosis_agent.py
from google.adk.agents import Agent

from shared.gateway import diagnosis_toolset
from shared.models import SONNET

SYSTEM_PROMPT = """You are the Diagnosis Agent for an electronics e-commerce support system.

Grounding rules (apply to every reply):
1. State only facts returned by your tools in this turn. Cite the specific source where
   relevant: knowledge-base article id, defect id, or web-search snippet.
2. Do not infer the cause from product brand or your own training data. If your tools
   returned no useful results, set probable_cause = "unknown" and confidence = "low".
3. Never invent defect ids, fix descriptions, or affected SKU lists. If a field is not
   in tool output, write "not found".

Procedure:
1. Search the knowledge base with the symptom and product name.
2. Check the known-defects database for the specific product SKU.
3. If both return no useful results, fall back to web_search once.
4. Pick the most likely cause from: firmware bug, hardware failure, user error,
   physical damage, unknown.
5. Return a structured diagnosis with these exact fields:
   probable_cause, defect_id (or "not found"), fix_type, fix_available (true/false),
   fix_description, confidence (high/medium/low), requires_hardware_assessment.

Do not recommend repairs, replacements, or returns. Your job is only to determine
what is wrong and whether a software fix exists. The fix_available flag is consumed
by the Cost & Recommendation Agent downstream."""

diagnosis_agent = Agent(
    name="diagnosis_agent",
    model=SONNET,
    description="Diagnoses technical issues using KB and known defect data.",
    instruction=SYSTEM_PROMPT,
    tools=[diagnosis_toolset()],
    output_key="diagnosis",
)


Overwriting agents/diagnosis_agent.py


In [13]:
%%writefile agents/cost_recommendation_agent.py
# workshop/agents/cost_recommendation_agent.py
from google.adk.agents import Agent

from shared.gateway import recommendation_toolset
from shared.models import SONNET
from tools.code_interpreter_tool import execute_python

SYSTEM_PROMPT = """You are the Cost & Recommendation Agent for an electronics e-commerce support system.

You are a TOOL called by an orchestrator. The orchestrator is the customer-facing
voice; you return structured analysis it will read and turn into a customer reply.
Do not address the customer. Do not write greetings, emojis, or markdown tables.

Two upstream specialists have replied earlier in conversation history:
1. customer_order_agent  -- verified customer/order summary (SKU, tier, etc.).
2. diagnosis_agent       -- technical diagnosis with probable_cause and fix_available.

Read these directly from history. If a needed field is missing, do not invent
it -- include a "gaps" entry in your output.

Grounding rules:
1. Use only numbers and facts from the upstream replies above or from the
   sandbox JSON output you produce. Never carry numbers from prior pipeline
   runs or your own knowledge.
2. Never invent options that are not backed by the upstream inputs.

Procedure:
1. Call ``get_repair_quote(product_sku, defect_type)`` using the SKU from the
   order summary and the defect_type from the diagnosis (probable_cause).
2. Build a Python script that defines an ``options`` list. Include exactly:
     - One paid-repair entry using the cost and estimated_days returned by
       ``get_repair_quote``.
     - One self-service firmware entry (cost=0.0, days=0) only if
       diagnosis.fix_available is true.
3. In that script, compute net cost and resolution days per option, rank by
   cost (ties: speed) and by speed (ties: cost), and select a recommendation:
   for tier == "premium" prefer the fastest zero-cost option; otherwise the
   lowest-cost option. Print one ``json.dumps(...)`` payload to stdout
   containing: all_options, fastest, lowest_cost, recommended,
   recommendation_reason.
4. Call ``execute_python(code, description="rank resolution options")``.
   Parse the JSON from the tool's return value (under ``stdout``).

Return value (your reply to the orchestrator) MUST be a single JSON object
with these exact keys:
  {
    "all_options": [...],
    "fastest": {...},
    "lowest_cost": {...},
    "recommended": {...},
    "recommendation_reason": "...",
    "excluded_options": [],
    "gaps": []
  }

Do not wrap this JSON in prose. Do not add markdown. If the sandbox returns
isError: true, return {"error": "sandbox_failed", "detail": "..."} and stop --
do not fabricate numbers."""

cost_recommendation_agent = Agent(
    name="cost_recommendation_agent",
    model=SONNET,
    description="Computes costs in the sandbox and returns a structured ranked recommendation as JSON.",
    instruction=SYSTEM_PROMPT,
    tools=[recommendation_toolset(), execute_python],
)


Overwriting agents/cost_recommendation_agent.py


### Step 11: First Check, Trace One Gateway Round-Trip End to End

The orchestrator pipeline is exercised end to end by Steps 12 and 13 (toolset enumeration and raw MCP `tools/list`). What this check does instead is run a single specialist -- `customer_order_agent`, the simplest of the three -- against the workshop customer message via `make_runner` + `run_turn`, with no orchestrator and no memory service in the loop, and walks the session event log to surface the MCP round-trip in three distinct layers:

1. **Layer 1**: the function_call events the agent emitted. These should be Gateway-prefixed names like `customer-order-tools___get_customer_profile`, proving the agent reached the tool through `McpToolset` rather than a stray local import.
2. **Layer 2**: the function_response events that came back. Each response is the Lambda return value (status + body), bubbled through Gateway and ADK's MCP client and dropped into the session as a tool result. Surfacing these proves the Lambda is reachable, the Bearer token authenticated, and the body is the same shape the local tools return.
3. **Layer 3**: the agent's final structured summary. With the upstream pieces verified, this last layer is just the model assembling its reply from real tool output.

The assertions at the bottom are deliberately strict: every tool call must carry the `customer-order-tools___` prefix (no silent fallback to local imports), at least one response must be non-empty, and the order's SKU from `data/orders.json` must appear in some response (a grounding check that the tool actually executed against bundled data inside the Lambda, not a hallucinated payload).

In [14]:
import json

from agents.customer_order_agent import customer_order_agent
from scenarios.scenario import APP_NAME, CUSTOMER_MESSAGE, EXPECTED_ORDER_ID, make_runner, run_turn

runner = make_runner(customer_order_agent)
session = await runner.session_service.create_session(app_name=APP_NAME, user_id="probe")
final_text = await run_turn(
    runner, user_id="probe", session_id=session.id, message=CUSTOMER_MESSAGE
)

# Re-fetch the session to walk the event log for tool calls and responses.
session = await runner.session_service.get_session(
    app_name=APP_NAME, user_id="probe", session_id=session.id
)

tool_calls: list[tuple[str, dict]] = []
tool_responses: list[tuple[str, object]] = []
for event in session.events:
    if not event.content or not event.content.parts:
        continue
    for part in event.content.parts:
        fc = getattr(part, "function_call", None)
        if fc:
            tool_calls.append((fc.name, dict(fc.args or {})))
        fr = getattr(part, "function_response", None)
        if fr:
            tool_responses.append((fr.name, fr.response))

print("=== Layer 1: tool calls the agent emitted ===")
for name, args in tool_calls:
    print(f"- {name}({json.dumps(args, default=str)})")

print("\n=== Layer 2: tool responses returned through Gateway -> Lambda ===")
for name, response in tool_responses:
    snippet = json.dumps(response, default=str)
    if len(snippet) > 400:
        snippet = snippet[:400] + "..."
    print(f"- {name} -> {snippet}")

print("\n=== Layer 3: agent's final summary ===")
print(final_text)

# Strict acceptance: prove the agent actually went through Gateway, not a local fallback.
GATEWAY_PREFIX = "customer-order-tools___"
non_prefixed = [name for name, _ in tool_calls if not name.startswith(GATEWAY_PREFIX)]
assert tool_calls, "customer_order_agent emitted no tool calls; Gateway path was never exercised."
assert not non_prefixed, (
    f"Some tool calls did not route through Gateway: {non_prefixed}. "
    "Expected every name to be prefixed with 'customer-order-tools___'."
)
assert tool_responses, "No function_response events in the session; Lambda did not return."
assert any(json.dumps(r, default=str).strip() for _, r in tool_responses), (
    "All tool responses were empty; Lambda returned nothing."
)
assert any(EXPECTED_ORDER_ID in json.dumps(r, default=str) for _, r in tool_responses), (
    f"None of the tool responses contained the expected order id {EXPECTED_ORDER_ID}; "
    "the Lambda may not be reading bundled data correctly."
)
assert final_text, "customer_order_agent produced no final text after the tool round-trip."

print(
    f"\nGateway round-trip auditable: {len(tool_calls)} call(s) routed through "
    f"'{GATEWAY_PREFIX}', {len(tool_responses)} Lambda response(s) returned, "
    "agent assembled a final summary from real tool output."
)

/opt/conda/lib/python3.12/site-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()
/opt/conda/lib/python3.12/site-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.BASE_AUTHENTICATED_TOOL is enabled.
  check_feature_enabled()
/opt/conda/lib/python3.12/site-packages/google/adk/models/llm_request.py:256: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


=== Layer 1: tool calls the agent emitted ===
- customer-order-tools___get_customer_profile({"customer_id": "C001"})
- customer-order-tools___get_order_details({"order_id": "ORD-2026-0342"})
- customer-order-tools___verify_purchase({"customer_id": "C001", "order_id": "ORD-2026-0342"})

=== Layer 2: tool responses returned through Gateway -> Lambda ===
- customer-order-tools___get_customer_profile -> {"content": [{"type": "text", "text": "{\"statusCode\":200,\"body\":\"{\\\"found\\\": true, \\\"customer_id\\\": \\\"C001\\\", \\\"name\\\": \\\"Alex Johnson\\\", \\\"account_tier\\\": \\\"premium\\\", \\\"email\\\": \\\"alex.johnson@example.com\\\"}\"}"}], "isError": false}
- customer-order-tools___get_order_details -> {"content": [{"type": "text", "text": "{\"statusCode\":200,\"body\":\"{\\\"found\\\": true, \\\"order_id\\\": \\\"ORD-2026-0342\\\", \\\"customer_id\\\": \\\"C001\\\", \\\"product_name\\\": \\\"Acme XGzG 15 NK30\\\", \\\"product_sku\\\": \\\"ACME-XGZG15-NK30-16GB-512SSD\\\",

### Step 12: Second Check, Eligibility Tools Are Registered But Not Yet Consumed

The Eligibility Agent does not exist in this notebook, since it is the attendee-written piece in the next one. What this notebook did do is register the `eligibility-tools` target on Gateway alongside the three in-use targets and expose an `eligibility_toolset()` factory. When the next notebook's attendees write `agents/eligibility_agent.py`, their `tools=[eligibility_toolset()]` line points at an already-running MCP endpoint.

This check verifies that pre-wiring: `eligibility_toolset()` returns a live `McpToolset`, and `await toolset.get_tools()` opens an MCP session, issues `tools/list`, and filters down to the four Gateway-prefixed tool names. If any of the four is missing, the next notebook would stall on infrastructure, so this check exists to catch that failure mode early.

We will add `eligibility_agent` as a fourth sibling `AgentTool` on the orchestrator in N6. The Gateway endpoint already exists; only the agent file is new.

In [15]:
from shared.gateway import eligibility_toolset

toolset = eligibility_toolset()
tools = await toolset.get_tools()

tool_names = [t.name for t in tools]
print(f"eligibility_toolset() exposed {len(tool_names)} tool(s):")
for name in tool_names:
    print(" -", name)

expected_eligibility_tools = {
    "eligibility-tools___check_warranty_coverage",
    "eligibility-tools___get_repair_options",
    "eligibility-tools___check_replacement_inventory",
    "eligibility-tools___get_return_eligibility",
}
missing = expected_eligibility_tools - set(tool_names)
assert not missing, (
    f"eligibility_toolset() is missing expected tools: {sorted(missing)}. "
    f"Notebook 5 attendees would fail to reach these tools."
)
print()
print("All four eligibility tools are reachable through Gateway — Notebook 5 pre-wiring confirmed.")

eligibility_toolset() exposed 4 tool(s):
 - eligibility-tools___check_replacement_inventory
 - eligibility-tools___check_warranty_coverage
 - eligibility-tools___get_repair_options
 - eligibility-tools___get_return_eligibility

All four eligibility tools are reachable through Gateway — Notebook 5 pre-wiring confirmed.


### Step 13: Third Check, Raw MCP `tools/list` Against the Gateway URL

The previous checks went through ADK's `McpToolset` and the live agent pipeline. This final check drops below both and speaks raw MCP JSON-RPC 2.0 to the Gateway using only `urllib.request`, proving the Gateway is reachable with the Cognito Bearer token from `shared/auth.py` and every registered `{target}___{tool}` is discoverable independently of ADK.

The MCP Streamable HTTP specification allows a server to return an `Mcp-Session-Id` header on `initialize` that the client then echoes on subsequent requests. The header is optional, so the helper below captures the ID if present and threads it through the `tools/list` call, which is what a production MCP client (including ADK's `McpToolset`) does under the hood.

All 11 registered target tools should come back: three from `customer-order-tools`, three from `diagnosis-tools`, four from `eligibility-tools`, and one from `recommendation-tools`. The Gateway also exposes a built-in semantic-search helper (`x_amz_bedrock_agentcore_search`), which we filter out of the assertion.

In [16]:
import json
import urllib.request

from shared.auth import get_cognito_token
from shared.config import get_ssm_parameter

gateway_url = get_ssm_parameter("gateway/gateway_url")
access_token = get_cognito_token()

# MCP session state. Populated by the initialize response and echoed on
# every subsequent request within the same session. The header name is
# "Mcp-Session-Id" per the MCP Streamable HTTP specification.
mcp_session_id: str | None = None


def _mcp_post(method: str, request_id: int, params: dict | None = None) -> dict:
    """POST a JSON-RPC 2.0 request to the Gateway and return the parsed response.

    On initialize, captures the Mcp-Session-Id response header so later calls
    can thread it back through -- the Gateway requires session continuity for
    everything after initialize.
    """
    global mcp_session_id

    payload = {"jsonrpc": "2.0", "id": request_id, "method": method}
    if params is not None:
        payload["params"] = params

    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream",
        "Authorization": f"Bearer {access_token}",
    }
    if mcp_session_id is not None:
        headers["Mcp-Session-Id"] = mcp_session_id

    request = urllib.request.Request(
        gateway_url,
        data=json.dumps(payload).encode("utf-8"),
        headers=headers,
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=30) as response:
        # Capture the session id from the initialize response so subsequent
        # requests can echo it back.
        if mcp_session_id is None:
            session_header = response.headers.get("Mcp-Session-Id")
            if session_header:
                mcp_session_id = session_header
        body = response.read().decode("utf-8")

    parsed = json.loads(body)
    if "error" in parsed:
        raise RuntimeError(
            f"MCP error [{parsed['error'].get('code')}]: "
            f"{parsed['error'].get('message')}"
        )
    return parsed


init_response = _mcp_post(
    "initialize",
    request_id=1,
    params={
        "protocolVersion": "2024-11-05",
        "capabilities": {},
        "clientInfo": {"name": "agentcore-workshop-n4", "version": "1.0.0"},
    },
)
print("initialize -> server:", init_response.get("result", {}).get("serverInfo", {}))
if mcp_session_id:
    print("Mcp-Session-Id captured and will be echoed on later calls:", mcp_session_id)
else:
    print("Gateway omitted Mcp-Session-Id on initialize; this server is stateless for this transport. Subsequent calls carry only the Bearer token.")

list_response = _mcp_post("tools/list", request_id=2)
raw_tool_names = [t["name"] for t in list_response.get("result", {}).get("tools", [])]
tool_names = [n for n in raw_tool_names if "x_amz_bedrock_agentcore" not in n]

print()
print(f"tools/list -> {len(tool_names)} target tool(s) (excluding semantic-search helper):")
for name in tool_names:
    print(" -", name)

expected_tools = {
    "customer-order-tools___get_customer_profile",
    "customer-order-tools___get_order_details",
    "customer-order-tools___verify_purchase",
    "diagnosis-tools___search_knowledge_base",
    "diagnosis-tools___get_known_defects",
    "diagnosis-tools___web_search",
    "eligibility-tools___check_warranty_coverage",
    "eligibility-tools___get_repair_options",
    "eligibility-tools___check_replacement_inventory",
    "eligibility-tools___get_return_eligibility",
    "recommendation-tools___get_repair_quote",
}
missing = expected_tools - set(tool_names)
assert not missing, f"Gateway tools/list is missing expected tools: {sorted(missing)}"
print()
print(f"All {len(expected_tools)} registered target tools are reachable over raw MCP JSON-RPC.")


initialize -> server: {'version': '1.0.0', 'name': 'CustomerSupportGateway'}
Gateway omitted Mcp-Session-Id on initialize; this server is stateless for this transport. Subsequent calls carry only the Bearer token.

tools/list -> 11 target tool(s) (excluding semantic-search helper):
 - customer-order-tools___get_customer_profile
 - customer-order-tools___get_order_details
 - customer-order-tools___verify_purchase
 - diagnosis-tools___get_known_defects
 - diagnosis-tools___search_knowledge_base
 - diagnosis-tools___web_search
 - eligibility-tools___check_replacement_inventory
 - eligibility-tools___check_warranty_coverage
 - eligibility-tools___get_repair_options
 - eligibility-tools___get_return_eligibility
 - recommendation-tools___get_repair_quote

All 11 registered target tools are reachable over raw MCP JSON-RPC.


### What You Built

- `shared/auth.py`, a 55-minute cached Cognito `client_credentials` token fetcher that reads client ID, secret, and token endpoint from SSM and returns a fresh Bearer token.
- `shared/gateway.py`, four `McpToolset` factory functions that point at the Gateway URL with a Bearer token plus a `tool_filter` restricting each toolset to its target's prefix.
- One Lambda fronting the four `tools/*.py` modules and four Gateway targets routing by name. The eligibility target is registered ahead of time even though no agent consumes it yet.
- Three rewired specialists. Customer/order, diagnosis, and cost & recommendation each swapped their local `tools=[func1, func2, ...]` list for `tools=[<agent>_toolset()]` (plus `execute_python` on the cost & recommendation agent, which stays a direct client call).
- Gateway is a transparent swap, since agents and orchestrator definitions don't change shape, only tool sources. The orchestrator still wires three sibling `AgentTool`s plus `preload_memory`, identical to N4 v2.
- Three acceptance checks passing: a 3-layer Gateway round-trip trace via `customer_order_agent` (function_calls -> Lambda responses -> agent summary), eligibility toolset enumeration, and raw MCP `tools/list` proving all 11 target tools are reachable independent of ADK.

Pre-wiring eligibility now makes the next notebook a pure agent-authoring exercise. The target is registered, the Lambda is deployed, and `eligibility_toolset()` is exported, so the attendee writes only `agents/eligibility_agent.py`.

### Cleanup

This notebook provisioned several persistent AWS resources. To remove them manually:

| Resource | How to delete |
|---|---|
| Cognito user pool and app client | `aws cognito-idp delete-user-pool-domain` then `delete-user-pool` (or use the Cognito console). |
| `CustomerSupportTools` Lambda and `CustomerSupportToolsLambdaRole` IAM role | `aws lambda delete-function --function-name CustomerSupportTools` and `aws iam delete-role` (after detaching policies). |
| `CustomerSupportGateway` and its four targets | Delete via the AgentCore console or `bedrock-agentcore-control:DeleteGatewayTarget` then `DeleteGateway`. |

`bash reset_env.sh --full` clears the SSM pointers under `/agentcore-workshop/cognito/*` and `/agentcore-workshop/gateway/*` but does not delete the underlying AWS resources.

### Next Up

**[Notebook 6: Build the Eligibility Agent](06_build_eligibility_agent.ipynb)** is the attendee-driven lab. You write `agents/eligibility_agent.py` and add it as a fourth sibling `AgentTool` on the orchestrator alongside `customer_order`, `diagnosis`, and `cost_recommendation`, with `preload_memory` remaining as the fifth tool, bringing the orchestrator's tool count to five. Diagnosis and eligibility run **concurrently** via Bedrock-native parallel tool calls in a single assistant turn (no `ParallelAgent` wrapper). Identity, Gateway, the Lambda, the `eligibility-tools` target, and `eligibility_toolset()` are all already wired, so that lab is pure ADK agent authoring with no AWS infrastructure work.